# Proyecto Final: Una Anomalía, Defendida
* **Integrantes:** Mateo Estay y Roberto Verdejo
* **Repositorio de GitHub:** https://github.com/matexx123/iele756_2026-6
* **Enlace al Video de la Defensa:** https://youtu.be/9CCdKEr8p8k?si=tyCtI6sWrQ-Dss6z

## 1. Declaración de la Anomalía: La Ilusión de Significancia del Desempleo

Luego de evaluar el pipline, se detecó una anomalía: la variable del porcentaje de desempleo (`pct_unemployed`) sufre una inversión completa de signo y una pérdida total de significancia estadística al cambiar la especificación del modelo de regresión. En el modelo Poisson, el desempleo muestra un impacto negativo y altamente significativo en las tasas de ENO (p-value de 0.0003), sugiriendo erróneamente un patrón de salud pública. Sin embargo, al controlar la varianza mediante una regresión Binomial Negativa, el efecto se vuelve positivo y completamente irrelevante (p-value de 0.999). Esta contradicción demuestra cómo un defecto estructural en los supuestos del modelo puede distorsionar por completo las conclusiones sobre el territorio.

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# 1. Cargar la tabla analítica usando la ruta absoluta especificada
ruta_csv = "C:/Users/Mateo/Downloads/tarea3_analytical_table (1).csv"
df = pd.read_csv(ruta_csv)

# 2. Limpiar la comuna de Santiago para evitar el colapso por datos corruptos
df_clean = df[df['nombre_comuna'] != 'Santiago']

# 3. Ejecutar los dos modelos rápidos de conteo con el offset de población
mod_poisson = smf.poisson("eno_total ~ pct_foreign + mean_schooling_chilean + pct_unemployed + dependency_ratio", data=df_clean, offset=np.log(df_clean['pop_total'])).fit(disp=0)
mod_negbin = smf.negativebinomial("eno_total ~ pct_foreign + mean_schooling_chilean + pct_unemployed + dependency_ratio", data=df_clean, offset=np.log(df_clean['pop_total'])).fit(disp=0)

# 4. Generar la Headline Figure (Tabla comparativa de la anomalía)
headline_figure = pd.DataFrame({
    "Coeficiente Poisson": mod_poisson.params,
    "P-Value Poisson": mod_poisson.pvalues,
    "Coeficiente Binomial Negativa": mod_negbin.params,
    "P-Value Binomial Negativa": mod_negbin.pvalues
}).loc[['pct_unemployed']]

print("--- FIGURA PRINCIPAL: IMPACTO DEL DESEMPLEO (SANTIAGO FILTRADO) ---")
display(headline_figure)

--- FIGURA PRINCIPAL: IMPACTO DEL DESEMPLEO (SANTIAGO FILTRADO) ---


,Coeficiente Poisson,P-Value Poisson,Coeficiente Binomial Negativa,P-Value Binomial Negativa
pct_unemployed,0.001865,0.000208,-0.000014,0.99865


In [4]:
# EXPLICACIÓN ALTERNATIVA 1: ¿Qué pasa si controlamos omitiendo también a San Pedro (el otro gran outlier)?
df_sin_outliers = df_clean[df_clean['nombre_comuna'] != 'San Pedro']
mod_nb_control2 = smf.negativebinomial("eno_total ~ pct_foreign + mean_schooling_chilean + pct_unemployed + dependency_ratio", data=df_sin_outliers, offset=np.log(df_sin_outliers['pop_total'])).fit(disp=0)

# EXPLICACIÓN ALTERNATIVA 2: Medición de la sobredispersión real del dataset limpio
mean_eno = df_clean['eno_total'].mean()
var_eno = df_clean['eno_total'].var()
sobredispersion = var_eno / mean_eno

print("--- VERIFICACIONES DE CONTROL ---")
print(f"P-value del Desempleo (Sin Santiago ni San Pedro): {mod_nb_control2.pvalues['pct_unemployed']:.4f}")
print(f"Razón de Sobredispersión real en la RM: {sobredispersion:.2f}")

--- VERIFICACIONES DE CONTROL ---
P-value del Desempleo (Sin Santiago ni San Pedro): 0.9397
Razón de Sobredispersión real en la RM: 1039.56


C:\Users\Mateo\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:3377: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
C:\Users\Mateo\anaconda3\Lib\site-packages\statsmodels\discrete\discrete_model.py:3377: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)


### Análisis de Alternativas

Con estas verificaciones se logró comprobar por qué ocurrió la anomalía. Primero, se quería descartar que el cambio de resultados fuera solo culpa de las comunas más extremas (como Santiago y San Pedro). Al sacarlas del análisis, el p-value del desempleo se mantuvo altísimo (0.9397). Como este valor está muy lejos del 0.05, confirmamos que el desempleo genuinamente no tiene efecto sobre las notificaciones de enfermedades, y que el problema no era un par de datos atípicos.

La verdadera razón de este "espejismo" es que los datos de la región son demasiado inestables: la variación es más de 1000 veces mayor al promedio. Como el modelo Poisson asume que los datos deben ser ordenados, no supo procesar esta diferencia y nos arrojó una relación matemática que en la realidad no existía. Al usar un modelo preparado para esta variación (Binomial Negativa), esa falsa importancia desaparece por completo.